## hierarchical BM

# only fix beta

In [ ]:
##### !/usr/bin/env python3
"""
单被试层级贝叶斯模型 - omega/beta固定版
1 CSV = 1被试的所有games
"""

import os
import sys
import gc
import psutil

# 环境配置
os.environ['PYTENSOR_FLAGS'] = 'floatX=float32,optimizer=fast_compile'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '4'

import numpy as np
import pandas as pd
import logging
from pathlib import Path

# JAX配置
try:
    import jax
    jax.config.update('jax_enable_x64', False)
    jax.config.update('jax_platform_name', 'cpu')
    import numpyro
    JAX_AVAILABLE = True
    print(f"JAX {jax.__version__} + NumPyro {numpyro.__version__}")
except ImportError:
    JAX_AVAILABLE = False
    print("JAX not installed")

import pymc as pm
import pytensor.tensor as pt
import arviz as az

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100

# 日志配置
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('/root/bhm_fitting_final.log')
    ]
)
logger = logging.getLogger(__name__)

DATA_PATH = Path("/root/seeg_behav/")
SAVE_PATH = Path("/root/results/bhm_final_fix_beta_26/")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

logger.info(f"Data path: {DATA_PATH}")
logger.info(f"Save path: {SAVE_PATH}")
logger.info(f"PyMC version: {pm.__version__}")
logger.info(f"JAX available: {JAX_AVAILABLE}")
logger.info(f"CPU cores: {os.cpu_count()}")


def preprocess_data(df_raw):
    """数据预处理"""
    df_filtered = df_raw[(df_raw['total_reward'] >= 0) | (df_raw['trial_general'] > 0)].copy()
    df_filtered.reset_index(drop=True, inplace=True)
    
    num_trials_per_game = 7
    num_games_estimated = len(df_filtered) // num_trials_per_game
    
    if len(df_filtered) == 0:
        return None
    
    results = []
    for game_num_est in range(num_games_estimated):
        cumulative_count_1 = 0
        cumulative_count_2 = 0
        cumulative_count_3 = 0
        
        start_idx = game_num_est * num_trials_per_game
        end_idx = start_idx + num_trials_per_game
        
        if end_idx > len(df_filtered):
            break
        
        for trial_in_game in range(num_trials_per_game):
            index = start_idx + trial_in_game
            
            chosen_deck_val = np.nan
            generated_number_1 = np.nan
            generated_number_2 = np.nan
            generated_number_3 = np.nan
            
            if trial_in_game < 6:
                if 'key_resp.keys' in df_filtered.columns:
                    chosen_deck_val = df_filtered.loc[index, 'key_resp.keys']
                if 'generated_number_1' in df_filtered.columns:
                    generated_number_1 = df_filtered.loc[index, 'generated_number_1']
                if 'generated_number_2' in df_filtered.columns:
                    generated_number_2 = df_filtered.loc[index, 'generated_number_2']
                if 'generated_number_3' in df_filtered.columns:
                    generated_number_3 = df_filtered.loc[index, 'generated_number_3']
            else:
                if 'key_resp_2.keys' in df_filtered.columns:
                    chosen_deck_val = df_filtered.loc[index, 'key_resp_2.keys']
                if 'true_reward_1' in df_filtered.columns:
                    generated_number_1 = df_filtered.loc[index, 'true_reward_1']
                if 'true_reward_2' in df_filtered.columns:
                    generated_number_2 = df_filtered.loc[index, 'true_reward_2']
                if 'true_reward_3' in df_filtered.columns:
                    generated_number_3 = df_filtered.loc[index, 'true_reward_3']
            
            try:
                if pd.notna(chosen_deck_val):
                    current_choice_int = int(float(chosen_deck_val))
                    if current_choice_int == 1:
                        cumulative_count_1 += 1
                    elif current_choice_int == 2:
                        cumulative_count_2 += 1
                    elif current_choice_int == 3:
                        cumulative_count_3 += 1
            except (ValueError, TypeError):
                pass
            
            condition_val = np.nan
            if 'trial_general' in df_filtered.columns and pd.notna(df_filtered.loc[index, 'trial_general']):
                condition_val = 1 if df_filtered.loc[index, 'trial_general'] < 7 else 2
            
            results.append({
                'Game': game_num_est + 1,
                'Trial': trial_in_game + 1,
                'Condition': condition_val,
                'Chosen Deck': chosen_deck_val,
                'Reward Card 1': generated_number_1,
                'Reward Card 2': generated_number_2,
                'Reward Card 3': generated_number_3,
                'Revealed Count Card 1': cumulative_count_1,
                'Revealed Count Card 2': cumulative_count_2,
                'Revealed Count Card 3': cumulative_count_3
            })
    
    return pd.DataFrame(results) if results else None


def extract_trial_data_for_bhm(df_processed, num_decks=3, num_force_trials=6, alpha_init=0.1):
    """提取trial数据"""
    num_total_games = df_processed['Game'].nunique()
    trial_data_list = []
    
    for game_idx in range(num_total_games):
        game_data = df_processed[df_processed['Game'] == game_idx + 1].sort_values('Trial').reset_index(drop=True)
        
        if len(game_data) < num_force_trials + 1:
            continue
        
        Q = np.zeros(num_decks)
        forced_choices = []
        forced_rewards = []
        
        for trial in range(num_force_trials):
            row = game_data.iloc[trial]
            
            try:
                deck_val = int(row['Chosen Deck'])
                if not (1 <= deck_val <= num_decks):
                    raise ValueError
                chosen_deck = deck_val - 1
            except (ValueError, TypeError):
                chosen_deck = int(np.argmax(Q))
            
            forced_choices.append(chosen_deck)
            rewards = [row[f'Reward Card {i+1}'] for i in range(num_decks)]
            forced_rewards.append(rewards)
            
            reward = rewards[chosen_deck]
            if pd.notna(reward):
                delta = reward - Q[chosen_deck]
                Q[chosen_deck] += alpha_init * delta
        
        free_trial_row = game_data.iloc[num_force_trials]
        try:
            deck_val_free = int(free_trial_row['Chosen Deck'])
            if not (1 <= deck_val_free <= num_decks):
                raise ValueError
            chosen_deck_free = deck_val_free - 1
        except (ValueError, TypeError):
            continue
        
        trial_data_list.append({
            'game_idx': game_idx,
            'forced_choices': np.array(forced_choices, dtype=np.int32),
            'forced_rewards': np.array(forced_rewards, dtype=np.float32),
            'free_choice': chosen_deck_free
        })
    
    return trial_data_list



def build_single_subject_model(trial_data_list, num_decks=3, num_force_trials=6):
    """
    单被试层级模型：beta固定，omega有game级扰动
    
    参数结构:
    - alpha: 被试基线 + 每个game的小扰动
    - gamma: 被试基线 + 每个game的小扰动
    - omega: 被试基线 + 每个game的小扰动 【新增】
    - beta: 被试固定值（所有games共享）
    """
    n_games = len(trial_data_list)
    
    if n_games == 0:
        return None, None
    
    logger.info(f"Building single-subject model with {n_games} games")
    
    all_forced_choices = pt.as_tensor_variable(
        np.array([d['forced_choices'] for d in trial_data_list], dtype=np.int32)
    )
    all_forced_rewards = pt.as_tensor_variable(
        np.array([d['forced_rewards'] for d in trial_data_list], dtype=np.float32)
    )
    all_free_choices = np.array([d['free_choice'] for d in trial_data_list], dtype=np.int32)
    
    with pm.Model() as model:
        # === 被试层参数（该被试的基线值） ===
        alpha_subj = pm.Beta('alpha_subj', alpha=2, beta=2)
        gamma_subj = pm.Gamma('gamma_subj', alpha=2, beta=2)
        omega_subj = pm.Gamma('omega_subj', alpha=2, beta=2)  # omega基线 【新增】
        beta_subj = pm.Gamma('beta_subj', alpha=1.5, beta=2)   # beta固定
        
        # === Game层参数（围绕被试基线变化） ===
        # alpha的game级变化
        alpha_game_raw = pm.Normal('alpha_game_raw', mu=0, sigma=1, shape=n_games)
        alpha_game_sigma = pm.HalfNormal('alpha_game_sigma', sigma=0.15)
        
        alpha_logit_base = pm.math.logit(alpha_subj)
        alpha_logit_games = alpha_logit_base + alpha_game_sigma * alpha_game_raw
        alpha_games = pm.Deterministic('alpha_games', pm.math.invlogit(alpha_logit_games))
        
        # gamma的game级变化
        gamma_game_raw = pm.Normal('gamma_game_raw', mu=0, sigma=1, shape=n_games)
        gamma_game_sigma = pm.HalfNormal('gamma_game_sigma', sigma=0.3)
        
        gamma_games = pm.Deterministic('gamma_games', 
            pm.math.exp(pm.math.log(gamma_subj) + gamma_game_sigma * gamma_game_raw)
        )
        
        # 【新增】omega的game级变化
        omega_game_raw = pm.Normal('omega_game_raw', mu=0, sigma=1, shape=n_games)
        omega_game_sigma = pm.HalfNormal('omega_game_sigma', sigma=0.3)
        
        omega_games = pm.Deterministic('omega_games',
            pm.math.exp(pm.math.log(omega_subj) + omega_game_sigma * omega_game_raw)
        )
        
        # beta在所有games中固定
        beta_games = pt.ones(n_games) * beta_subj
        
        # === Q和I计算（保持不变） ===
        def compute_Q_I_single_game(game_idx):
            forced_choices_g = all_forced_choices[game_idx]
            forced_rewards_g = all_forced_rewards[game_idx]
            alpha_g = alpha_games[game_idx]
            
            Q = pt.zeros(num_decks)
            I = pt.zeros(num_decks)
            
            for step in range(num_force_trials):
                choice_idx = forced_choices_g[step]
                reward = forced_rewards_g[step, choice_idx]
                
                is_valid = ~pt.isnan(reward)
                delta = reward - Q[choice_idx]
                Q = pt.set_subtensor(
                    Q[choice_idx],
                    Q[choice_idx] + pt.switch(is_valid, alpha_g * delta, 0.0)
                )
                I = pt.set_subtensor(I[choice_idx], I[choice_idx] + 1)
            
            return Q, I
        
        Q_all = []
        I_all = []
        for i in range(n_games):
            Q, I = compute_Q_I_single_game(i)
            Q_all.append(Q)
            I_all.append(I)
        
        Q_all = pt.stack(Q_all)
        I_all = pt.stack(I_all)
        
        # === V和Softmax计算 ===
        gamma_safe = pt.clip(gamma_games[:, None], 0.05, 2.0)
        I_transformed = pt.pow(I_all, gamma_safe)
        
        # 
        omega_safe = pt.clip(omega_games[:, None], 0.01, 5.0)
        V_all = Q_all - omega_safe * I_transformed
        
        V_safe = pt.clip(V_all, -8, 8)
        beta_safe = pt.clip(beta_games[:, None], 0.1, 5)
        
        V_shifted = V_safe - pt.max(V_safe, axis=1, keepdims=True)
        scaled_V = pt.clip(beta_safe * V_shifted, -15, 15)
        
        exp_V = pt.exp(scaled_V)
        probs = exp_V / (pt.sum(exp_V, axis=1, keepdims=True) + 1e-8)
        probs = pt.clip(probs, 1e-8, 1 - 1e-8)
        
        pm.Categorical('choices', p=probs, observed=all_free_choices)
        
        logger.info("Model built, starting sampling...")
        
        # === 采样（JAX优先） ===
        use_jax = JAX_AVAILABLE
        trace = None
        
        if use_jax:
            try:
                from pymc.sampling.jax import sample_numpyro_nuts
                
                logger.info("Using NumPyro (JAX) sampler...")
                trace = sample_numpyro_nuts(
                    draws=3000,
                    tune=1500,
                    chains=4,
                    target_accept=0.98,
                    random_seed=42,
                    chain_method='vectorized',
                    idata_kwargs={'log_likelihood': True}
                )
                
                logger.info("✓ JAX sampling completed")
                
            except Exception as e:
                logger.warning(f"JAX sampling failed: {e}")
                logger.info("Falling back to PyMC...")
                use_jax = False
        
        if not use_jax:
            logger.info("Using PyMC NUTS...")
            trace = pm.sample(
                draws=3000,
                tune=1500,
                chains=4,
                cores=4,
                return_inferencedata=True,
                target_accept=0.98,
                max_treedepth=12,
                random_seed=42,
                idata_kwargs={'log_likelihood': True}
            )
        
        # === 质量检查 ===
        if trace is not None:
            n_divs = trace.sample_stats['diverging'].sum().item()
            total_draws = len(trace.posterior.chain) * len(trace.posterior.draw)
            div_pct = n_divs / total_draws * 100
            
            logger.info(f"Sampling complete: {n_divs} divergences ({div_pct:.1f}%)")
            
            # 检查关键参数
            summary = az.summary(trace, var_names=['alpha_subj', 'gamma_subj', 'omega_subj', 'beta_subj'])
            max_rhat = summary['r_hat'].max()
            min_ess = summary['ess_bulk'].min()
            
            logger.info(f"Quality check: max_rhat={max_rhat:.3f}, min_ess={min_ess:.0f}")
            
            if max_rhat > 1.05 or min_ess < 100:
                logger.warning("采样质量不佳")
            else:
                logger.info("采样质量良好")
    
    del model
    gc.collect()
    
    return None, trace


def extract_game_metrics_single_subject(trace, trial_data_list, df_processed, 
                                        num_decks=3, num_force_trials=6):
    """
    从单被试模型提取game级别指标（修改版：omega现在是game级别）
    """
    posterior_mean = trace.posterior.mean(dim=['chain', 'draw'])
    
    # 被试固定参数（只有beta）
    beta_subj = float(posterior_mean['beta_subj'].values)
    
    # 被试基线参数（用于参考）
    omega_subj_baseline = float(posterior_mean['omega_subj'].values)
    
    # Game级别参数（现在包括omega）【修改】
    alpha_games_raw = posterior_mean['alpha_games'].values
    gamma_games_raw = posterior_mean['gamma_games'].values
    omega_games_raw = posterior_mean['omega_games'].values  # 【新增】
    
    n_expected = len(trial_data_list)
    
    # 处理scalar退化
    if alpha_games_raw.ndim == 0:
        logger.warning(f"参数退化为scalar，扩展为{n_expected}个值")
        alpha_games = np.full(n_expected, float(alpha_games_raw))
        gamma_games = np.full(n_expected, float(gamma_games_raw))
        omega_games = np.full(n_expected, float(omega_games_raw))  # 【新增】
    else:
        alpha_games = alpha_games_raw
        gamma_games = gamma_games_raw
        omega_games = omega_games_raw  # 【新增】
    
    # 维度验证
    logger.info(f"Expected: {n_expected}, alpha: {alpha_games.shape}, gamma: {gamma_games.shape}, omega: {omega_games.shape}")
    
    if len(alpha_games) != n_expected:
        logger.error(f"维度不匹配！alpha={len(alpha_games)} vs expected={n_expected}")
        if len(alpha_games) == 1:
            alpha_games = np.full(n_expected, alpha_games[0])
            gamma_games = np.full(n_expected, gamma_games[0])
            omega_games = np.full(n_expected, omega_games[0])  # 【新增】
        elif len(alpha_games) < n_expected:
            return pd.DataFrame()
        else:
            alpha_games = alpha_games[:n_expected]
            gamma_games = gamma_games[:n_expected]
            omega_games = omega_games[:n_expected]  # 【新增】
    
    # 预建索引映射
    trial_idx_map = {td['game_idx']: idx for idx, td in enumerate(trial_data_list)}
    
    results = []
    n_games = df_processed['Game'].nunique()
    
    for game_idx in range(n_games):
        game_data = df_processed[df_processed['Game'] == game_idx + 1].sort_values('Trial').reset_index(drop=True)
        
        if len(game_data) < num_force_trials + 1:
            continue
        
        trial_idx = trial_idx_map.get(game_idx)
        
        if trial_idx is None or trial_idx >= len(alpha_games):
            logger.warning(f"Game {game_idx+1} 无有效索引")
            continue
        
        try:
            alpha_g = float(alpha_games[trial_idx])
            gamma_g = float(gamma_games[trial_idx])
            omega_g = float(omega_games[trial_idx])  # 【修改：现在从game级别获取】
            beta_g = beta_subj  # 保持固定
        except (IndexError, TypeError) as e:
            logger.error(f"Game {game_idx+1} 参数提取失败: {e}")
            continue
        
        # Q和I计算（保持不变）
        Q = np.zeros(num_decks, dtype=np.float64)
        I = np.zeros(num_decks, dtype=np.float64)
        
        for trial_in_game in range(num_force_trials):
            row = game_data.iloc[trial_in_game]
            
            try:
                chosen_deck = int(row['Chosen Deck']) - 1
                if not (0 <= chosen_deck < num_decks):
                    chosen_deck = int(np.argmax(Q))
            except:
                chosen_deck = int(np.argmax(Q))
            
            reward = row[f'Reward Card {chosen_deck + 1}']
            
            if pd.notna(reward) and np.isfinite(reward):
                delta = reward - Q[chosen_deck]
                Q[chosen_deck] += alpha_g * delta
            
            I[chosen_deck] += 1
        
        # V和probs计算
        gamma_clipped = np.clip(gamma_g, 0.05, 2.0)
        I_transformed = np.power(I, gamma_clipped)
        
        V = Q - omega_g * I_transformed  # 使用game级别的omega
        
        V_stable = V - np.max(V)
        exp_V = np.exp(beta_g * V_stable)
        
        if np.any(np.isnan(exp_V)) or np.any(np.isinf(exp_V)):
            logger.warning(f"Game {game_idx+1} exp_V异常")
            probs = np.ones(num_decks) / num_decks
        else:
            probs = exp_V / (np.sum(exp_V) + 1e-10)
            probs = np.clip(probs, 1e-10, 1 - 1e-10)
            if not np.isclose(probs.sum(), 1.0, atol=1e-6):
                probs = probs / probs.sum()
        
        # 获取实际选择
        free_trial_row = game_data.iloc[num_force_trials]
        try:
            chosen_deck_free = int(free_trial_row['Chosen Deck']) - 1
            if not (0 <= chosen_deck_free < num_decks):
                chosen_deck_free = int(np.argmax(probs))
        except:
            chosen_deck_free = int(np.argmax(probs))
        
        chosen_prob = float(probs[chosen_deck_free])
        
        results.append({
            'Game': game_idx + 1,
            'alpha': alpha_g,
            'gamma': gamma_g,
            'omega': omega_g,  # 【修改：现在是game级别的值】
            'beta': beta_g,
            'Q_Deck_1': float(Q[0]),
            'Q_Deck_2': float(Q[1]),
            'Q_Deck_3': float(Q[2]),
            'I_Deck_1': float(I[0]),
            'I_Deck_2': float(I[1]),
            'I_Deck_3': float(I[2]),
            'I_transformed_Deck_1': float(I_transformed[0]),
            'I_transformed_Deck_2': float(I_transformed[1]),
            'I_transformed_Deck_3': float(I_transformed[2]),
            'V_Deck_1': float(V[0]),
            'V_Deck_2': float(V[1]),
            'V_Deck_3': float(V[2]),
            'Softmax_Prob_Deck_1': float(probs[0]),
            'Softmax_Prob_Deck_2': float(probs[1]),
            'Softmax_Prob_Deck_3': float(probs[2]),
            'Chosen_Deck': chosen_deck_free + 1,
            'Chosen_Prob': chosen_prob
        })
    
    logger.info(f"成功提取 {len(results)} 个game的指标")
    return pd.DataFrame(results)





def main():
    logger.info("="*70)
    logger.info("开始单被试层级贝叶斯模型拟合")
    logger.info("="*70)
    
    csv_files = list(DATA_PATH.glob("*.csv"))
    #csv_files = [f for f in csv_files if f.name in ['10.csv', '11.csv']]
    logger.info(f"找到 {len(csv_files)} 个CSV文件")
    
    if not csv_files:
        logger.warning(f"在 {DATA_PATH} 中未找到CSV文件！")
        return
    
    successful = 0
    failed = 0
    
    for idx, file_path in enumerate(sorted(csv_files)):
        file_name = file_path.name
        
        if file_name.startswith("._"):
            continue
        
        logger.info("="*70)
        logger.info(f"处理文件 {idx+1}/{len(csv_files)}: {file_name}")
        logger.info("="*70)
        
        process = psutil.Process()
        mem_before = process.memory_info().rss / 1024**3
        logger.info(f"内存使用: {mem_before:.1f}GB")
        
        try:
            # 数据加载
            try:
                df_raw = pd.read_csv(file_path, encoding='utf-8-sig')
            except UnicodeDecodeError:
                df_raw = pd.read_csv(file_path, encoding='latin1')
            
            logger.info(f"数据加载成功, 共 {len(df_raw)} 行")
            
            # 预处理
            processed_data = preprocess_data(df_raw)
            if processed_data is None:
                logger.warning(f"预处理失败: {file_name}")
                failed += 1
                continue
            
            logger.info(f"预处理成功, 共 {len(processed_data)} 行")
            
            # 提取trial数据
            trial_data_list = extract_trial_data_for_bhm(processed_data)
            if not trial_data_list:
                logger.warning(f"未找到有效的trial数据: {file_name}")
                failed += 1
                continue
            
            logger.info(f"提取trial数据成功, 共 {len(trial_data_list)} 个game")
            
            # 模型拟合
            model, trace = build_single_subject_model(trial_data_list)
            
            if trace is None:
                logger.warning(f"模型构建失败: {file_name}")
                failed += 1
                continue
            
            base_name = file_path.stem
            
            # 提取game级别指标
            logger.info("提取game级别指标...")
            game_metrics = extract_game_metrics_single_subject(
                trace, trial_data_list, processed_data
            )
            
            game_metrics.to_csv(SAVE_PATH / f"{base_name}_game_metrics.csv", index=False)
            logger.info(f"Game指标已保存: {len(game_metrics)} games")
            
            # 保存被试层参数
            subj_summary = az.summary(
                trace,
                var_names=['alpha_subj', 'gamma_subj', 'omega_subj', 'beta_subj']
            )
            subj_summary.to_csv(SAVE_PATH / f"{base_name}_subject_params.csv")
            logger.info(f"被试参数已保存")
            
            # 保存trace
            trace.to_netcdf(SAVE_PATH / f"{base_name}_trace.nc")
            logger.info(f"Trace数据已保存")
            
            # 绘图
            az.plot_trace(trace, var_names=['alpha_subj', 'gamma_subj', 'omega_subj', 'beta_subj'])
            plt.tight_layout()
            plt.savefig(SAVE_PATH / f"{base_name}_trace.png", dpi=100)
            plt.close()
            
            az.plot_posterior(trace, var_names=['alpha_subj', 'gamma_subj', 'omega_subj', 'beta_subj'])
            plt.tight_layout()
            plt.savefig(SAVE_PATH / f"{base_name}_posterior.png", dpi=100)
            plt.close()
            
            logger.info(f"✓ {file_name} 拟合完成!")
            successful += 1
            
            # 释放内存
            del trace, processed_data, trial_data_list, game_metrics
            gc.collect()
            
            if (idx + 1) % 3 == 0:
                logger.info("执行深度清理...")
                gc.collect()
                mem_after = process.memory_info().rss / 1024**3
                logger.info(f"清理后内存: {mem_after:.1f}GB")
            
        except Exception as e:
            logger.error(f"✗ {file_name} 拟合失败: {str(e)}", exc_info=True)
            failed += 1
            gc.collect()
            continue
    
    logger.info("="*70)
    logger.info(f"拟合完成! 成功: {successful}, 失败: {failed}")
    logger.info("="*70)


if __name__ == "__main__":
    main()
